# Notebook d'Inférence - Prédictions Consommation Électrique

Ce notebook permet de faire des prédictions de consommation électrique pour la journée suivante.

## Fonctionnalités :
1. Chargement du modèle depuis MLflow
2. Génération des features pour la journée suivante (toutes les 30 minutes)
3. Prédictions sur toutes les plages horaires
4. Stockage des prédictions en base de données

## Structure de la table de stockage :
- `prediction_id`: TEXT (UUID)
- `entity_id`: BIGINT
- `model_name`: TEXT
- `model_version`: TEXT
- `prediction`: DOUBLE PRECISION
- `prediction_proba`: TEXT
- `prediction_timestamp`: TIMESTAMP
- `features_hash`: TEXT
- `run_id`: TEXT
- `ground_truth`: BIGINT

## 0. Initialisation et Configuration

In [8]:
import os
import logging
import warnings
import uuid
import hashlib
from datetime import datetime, timedelta
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
import pandas as pd
import numpy as np

# Charger les variables d'environnement
load_dotenv(find_dotenv(".env"), override=True)
load_dotenv(find_dotenv(".env.secrets"), override=True)

# Configurer le logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Ignorer certains warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pandas")

print("✅ Initialisation terminée")

✅ Initialisation terminée


## 1. Configuration de la Base de Données

In [9]:
# Configuration PostgreSQL (à adapter selon votre environnement)
DB_URI = os.getenv("PREDICTIONS_POSTGRES_URI")

PREDICTIONS_TABLE = os.getenv("PREDICTIONS_TABLE", "predictions")

if DB_URI:
    print("📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI")
    print(f"  Table: {PREDICTIONS_TABLE}")
else:
    DB_CONFIG = {
        "host": os.getenv("DB_HOST", "localhost"),
        "port": os.getenv("DB_PORT", "5432"),
        "database": os.getenv("DB_NAME", "jinsudai"),
        "user": os.getenv("DB_USER", "postgres"),
        "password": os.getenv("DB_PASSWORD", ""),
    }
    print(f"📊 Configuration DB :")
    print(f"  Host: {DB_CONFIG['host']}")
    print(f"  Port: {DB_CONFIG['port']}")
    print(f"  Database: {DB_CONFIG['database']}")
    print(f"  Table: {PREDICTIONS_TABLE}")

📌 Utilisation de l'URI PostgreSQL : PREDICTIONS_POSTGRES_URI
  Table: predictions


## 2. Configuration MLflow

In [10]:
## 2. Configuration MLflow
import mlflow
import sys
sys.path.insert(0, str(Path.cwd() / "src"))

from ml.utils.pipelines import PredictionPipeline

# Configuration MLflow
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "https://jinsudai-mlflow.hf.space/")
MODEL_NAME = os.getenv("MODEL_NAME", "consumption_model_test")
MODEL_ALIAS = os.getenv("MODEL_ALIAS", "prod")  # 'prod' ou 'staging'

pipeline = PredictionPipeline(
    mlflow_uri=MLFLOW_TRACKING_URI,
    experiment_name=None,
    db_uri=DB_URI
)

pipeline_ready = pipeline.setup()

print("🔧 Pipeline d'inférence initialisé")
print(f"  Modèle: {MODEL_NAME}")
print(f"  Alias: {MODEL_ALIAS}")
print(f"  Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"  Pipeline prêt: {pipeline_ready}")


2026-06-11 18:41:11,153 - ml.utils.pipelines.Prediction_pipeline - INFO - Pipeline d'inférence initialisé
2026-06-11 18:41:11,156 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 1: CONFIGURATION ===


2026-06-11 18:41:12,071 - ml.utils.models.inference_model - INFO - MLflow connecté à https://jinsudai-mlflow.hf.space/
2026-06-11 18:41:12,072 - ml.utils.pipelines.Prediction_pipeline - INFO - MLflow configuré
2026-06-11 18:41:12,281 - ml.utils.pipelines.database_handler - INFO - Connexion à la base de données vérifiée
2026-06-11 18:41:12,447 - ml.utils.pipelines.database_handler - INFO - Table predictions_pipeline créée ou déjà existante
2026-06-11 18:41:12,449 - ml.utils.pipelines.Prediction_pipeline - INFO - Base de données configurée


🔧 Pipeline d'inférence initialisé
  Modèle: consumption_model_test
  Alias: prod
  Tracking URI: https://jinsudai-mlflow.hf.space/
  Pipeline prêt: True


## 3. Chargement du Modèle depuis MLflow

In [11]:
print(f"🔄 Chargement du modèle {MODEL_NAME} (alias: {MODEL_ALIAS})...")

try:
    if not pipeline.load_model(MODEL_NAME, alias_prod=MODEL_ALIAS):
        raise RuntimeError("Échec du chargement du modèle via le pipeline")

    model_info = pipeline.get_model_info() or {}
    model_version = model_info.get("version", "unknown")

    print(f"✅ Modèle chargé avec succès")
    print(f"  Version: {model_version}")

except Exception as e:
    logger.error(f"Erreur lors du chargement du modèle: {e}")
    raise

2026-06-11 18:41:12,486 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 2: CHARGEMENT DU MODÈLE ===


🔄 Chargement du modèle consumption_model_test (alias: prod)...


2026-06-11 18:41:13,715 - ml.utils.models.inference_model - INFO - Model registry source: runs:/49d0a71fbb044d768ce593d01a3a1096/model
2026/06/11 18:41:20 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at s3://mlflow-artifacts/2/49d0a71fbb044d768ce593d01a3a1096/artifacts/model. Returning destination path.
2026-06-11 18:41:47,137 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: gzyifihhbrqgqcynlshj.storage.supabase.co. Connection pool size: 10
2026-06-11 18:41:49,183 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: gzyifihhbrqgqcynlshj.storage.supabase.co. Connection pool size: 10
2026-06-11 18:41:49,543 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: gzyifihhbrqgqcynlshj.storage.supabase.co. Connection pool size: 10
2026-06-11 18:41:49,603 - ml.utils.models.inference_model - WARNING - Chargement sklearn échoué (Could not find a registered artifact repos

✅ Modèle chargé avec succès
  Version: 12


## 4. Génération des Features pour la Journée Suivante

In [14]:

features_df = pipeline.generate_data()


2026-06-11 18:47:21,065 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 3: GÉNÉRATION DES DONNÉES ===
2026-06-11 18:47:21,090 - ml.utils.pipelines.data_generator - INFO - Données d'inférence générées: 72 échantillons


## 5. Préparation des Features pour le Modèle

In [ ]:

# Préparer les features via PredictionPipeline
features_prep = pipeline.prepare_features()

print(f"✅ Features préparées via PredictionPipeline: {features_prep.shape}")
display(features_prep.head())

2026-06-11 15:10:55,058 - ml.utils.pipelines.Prediction_pipeline - INFO - DataFrame d'inférence chargé: (48, 8)
2026-06-11 15:10:55,062 - ml.utils.pipelines.data_generator - ERROR - Colonne manquante lors de la préparation des features: "['Horodate', 'jour_de_la_semaine', 'jour_férié'] not in index"
2026-06-11 15:10:55,064 - ml.utils.pipelines.Prediction_pipeline - ERROR - Impossible de préparer les features


AttributeError: 'NoneType' object has no attribute 'head'

## 6. Prédictions

In [ ]:
print("🔄 Prédictions en cours...")

try:
    if not pipeline_ready:
        raise RuntimeError("Le pipeline d'inférence n'est pas prêt")

    prediction_ok = pipeline.run_predictions(feature_columns)
    if not prediction_ok:
        raise RuntimeError("Échec des prédictions via le pipeline")

    predictions_df = pipeline.get_predictions_df()
    if predictions_df is None or predictions_df.empty:
        raise RuntimeError("Aucune prédiction générée")

    print(f"✅ Prédictions terminées: {len(predictions_df)} valeurs")
    print()
    print("📊 Statistiques des prédictions:")
    print(f"  Min: {predictions_df['prediction'].min():.2f}")
    print(f"  Max: {predictions_df['prediction'].max():.2f}")
    print(f"  Mean: {predictions_df['prediction'].mean():.2f}")
    print(f"  Std: {predictions_df['prediction'].std():.2f}")

    if "confidence" in predictions_df.columns:
        print()
        print(f"📈 Scores de confiance disponibles: {predictions_df['confidence'].shape}")

    display(predictions_df.head())
except Exception as e:
    logger.error(f"Erreur lors des prédictions: {e}")
    raise


2026-06-11 14:37:57,388 - ml.utils.pipelines.Prediction_pipeline - INFO - === ÉTAPE 2: CHARGEMENT DU MODÈLE ===


🔄 Prédictions en cours...


2026-06-11 14:37:58,399 - ml.utils.models.inference_model - INFO - Model registry source: runs:/49d0a71fbb044d768ce593d01a3a1096/model
2026/06/11 14:38:01 INFO mlflow.store.artifact.artifact_repo: No artifacts found to download at s3://mlflow-artifacts/2/49d0a71fbb044d768ce593d01a3a1096/artifacts/model. Returning destination path.
2026-06-11 14:38:21,127 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: gzyifihhbrqgqcynlshj.storage.supabase.co. Connection pool size: 10
2026-06-11 14:38:22,171 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: gzyifihhbrqgqcynlshj.storage.supabase.co. Connection pool size: 10
2026-06-11 14:38:22,743 - urllib3.connectionpool - WARNING - Connection pool is full, discarding connection: gzyifihhbrqgqcynlshj.storage.supabase.co. Connection pool size: 10
2026-06-11 14:38:23,374 - ml.utils.models.inference_model - WARNING - Chargement sklearn échoué (Could not find a registered artifact repos

RuntimeError: Échec des prédictions via le pipeline

## 7. Préparation des Données pour Stockage

In [ ]:
predictions_df = pipeline.get_predictions_df()

if predictions_df is None or predictions_df.empty:
    raise RuntimeError("Aucune prédiction disponible pour le stockage")

print(f"✅ Données prêtes pour stockage: {len(predictions_df)} enregistrements")
display(predictions_df.head())

## 8. Connexion à la Base de Données

In [ ]:
if DB_URI:
    print("📌 Le pipeline stockera les prédictions dans la base de données via PredictionPipeline.")
else:
    print("⚠️ Aucune URI DB fournie : le stockage via PredictionPipeline sera ignoré.")

In [ ]:
print("🔄 Stockage des prédictions en cours via PredictionPipeline...")

try:
    if not pipeline_ready:
        raise RuntimeError("Le pipeline d'inférence n'est pas prêt")

    stored = pipeline.store_predictions()
    if not stored:
        raise RuntimeError("Échec du stockage des prédictions via le pipeline")

    print("✅ Prédictions stockées via PredictionPipeline")
except Exception as e:
    logger.error(f"Erreur lors du stockage des prédictions: {e}")
    raise


In [ ]:
# Le stockage des prédictions est maintenant entièrement géré par PredictionPipeline.
# Ce notebook n'utilise plus d'insertion manuelle en base de données.

## 10. Nettoyage

In [ ]:
# Aucune connexion manuelle à fermer : PredictionPipeline gère la persistance et la connexion.
print("
🎉 Pipeline d'inférence terminé avec succès")


## 11. Résumé

In [ ]:
print("=" * 50)
print("RÉSUMÉ DU PIPELINE D'INFÉRENCE")
print("=" * 50)
print("
📊 Modèle:")
print(f"  Nom: {MODEL_NAME}")
print(f"  Version: {model_version}")
print("
⏰ Prédictions:")
prediction_date = None
if "prediction_timestamp" in predictions_df.columns:
    prediction_date = predictions_df['prediction_timestamp']
elif "Horodate" in predictions_df.columns:
    prediction_date = predictions_df['Horodate']

prediction_date_label = (
    prediction_date.dt.date.iloc[0] if prediction_date is not None else "inconnue"
)
print(f"  Date cible: {prediction_date_label}")
print(f"  Nombre de plages horaires: {len(predictions_df)}")
print(f"  Fréquence: 30 minutes")
print("
📈 Statistiques:")
print(f"  Min: {predictions_df['prediction'].min():.2f}")
print(f"  Max: {predictions_df['prediction'].max():.2f}")
print(f"  Mean: {predictions_df['prediction'].mean():.2f}")
print("
💾 Stockage:")
if DB_URI:
    print(f"  Statut: Stocké via PredictionPipeline")
else:
    print(f"  Statut: Non stocké (URI DB absente)")
print("=" * 50)
